LOZO Medium-Scale Replication (Prime Intellect, 80GB)

This notebook reproduces the LOZO medium-model pipeline in a week8-style, script-centric workflow:

1. **Environment + repo setup** (Prime-friendly paths and CUDA checks)
2. **Data preparation** (k-shot splits for K=16 and K=512)
3. **Smoke suite** (quick LOZO-only sanity check)
4. **Full suite** (multi-task, multi-seed LOZO runs)
5. **Aggregation + plots** (LOZO performance + memory figures)

Primary reference code: [optsuite/LOZO](https://github.com/optsuite/LOZO) (`medium_models`).

---
## §0 — Environment and Runtime Profile

This section defines reproducible paths and checks GPU readiness for a single 80GB Prime Intellect machine.

**Assumptions**
- Run from the `stat4830` repo root.
- Environment is prepared with `uv sync`.
- One CUDA GPU is available (80GB target profile).

In [ ]:
from __future__ import annotations

import json
import os
import select
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt


def detect_repo_root(start: Path) -> Path:
    """Find stat4830 repo root even when notebook starts in ./notebooks."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not detect repo root. Start Jupyter from the stat4830 root or set REPO_ROOT manually."
    )


REPO_ROOT = detect_repo_root(Path.cwd().resolve())
LOZO_ROOT = REPO_ROOT / "external" / "LOZO"
MEDIUM_DIR = LOZO_ROOT / "medium_models"
LOZO_PINNED_REF = "5d1ade5bf41846471d33ba526f877829b9a19856"

RUN_TAG = "lozo_medium_prime80gb"
SMOKE_RUN_ROOT = REPO_ROOT / "results" / RUN_TAG / "smoke"
FULL_RUN_ROOT = REPO_ROOT / "results" / RUN_TAG / "full"

TASKS_FULL = ["SST-2", "sst-5", "SNLI", "MNLI", "RTE", "trec"]
SEEDS_FULL = [13, 21, 42, 87, 100]
K_VALUES_FULL = [16, 512]

print("cwd:", Path.cwd())
print("REPO_ROOT:", REPO_ROOT)
print("LOZO_ROOT:", LOZO_ROOT)
print("LOZO_PINNED_REF:", LOZO_PINNED_REF)
print("RUN_TAG:", RUN_TAG)


def _safe_read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def _marker_progress(run_root: Path) -> tuple[int, int]:
    manifest = _safe_read_json(run_root / "manifest.json")
    runs = manifest.get("runs", [])
    total = len(runs)
    done = 0
    for spec in runs:
        result_dir = Path(str(spec.get("result_dir", "")))
        task = str(spec.get("task", ""))
        if not task:
            continue
        marker = result_dir / f"test_results_{task}.txt"
        if marker.exists():
            done += 1
    return done, total


def print_training_activity_check(run_root: Path, label: str) -> None:
    summary_path = run_root / "summary.json"
    payload = _safe_read_json(summary_path)
    if not payload:
        print(f"[{label}] no summary yet at {summary_path}")
        return

    counts = payload.get("counts", {})
    total_runs = payload.get("num_runs", 0)
    print(f"[{label}] final counts: {counts} / total_runs={total_runs}")

    records = payload.get("records", [])
    metric_records = [r for r in records if r.get("metrics")]
    if metric_records:
        latest = metric_records[-1]
        print(
            f"[{label}] latest metric record -> "
            f"task={latest.get('task')} k={latest.get('k')} seed={latest.get('seed')} "
            f"status={latest.get('status')} metrics={latest.get('metrics')}"
        )
        print(f"[{label}] latest log_path: {latest.get('log_path')}")
        print(f"[{label}] latest result_dir: {latest.get('result_dir')}")
    else:
        print(f"[{label}] no metric-bearing records found in summary yet.")


def run_suite_with_activity(
    cmd: list[str],
    run_root: Path,
    label: str,
    heartbeat_sec: int = 30,
) -> int:
    run_root = Path(run_root)
    started = time.time()
    print(f"[{label}] launch command:")
    print(" ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    active_run = ""
    active_log_path: Path | None = None
    last_log_size = -1
    last_heartbeat = time.time()

    assert proc.stdout is not None
    while True:
        ready, _, _ = select.select([proc.stdout], [], [], 1.0)
        if ready:
            line = proc.stdout.readline()
            if line:
                print(line, end="")
                stripped = line.strip()
                if stripped.startswith("[") and "]" in stripped:
                    active_run = stripped
                if stripped.startswith("log_path="):
                    active_log_path = Path(stripped.split("=", 1)[1])
                    last_log_size = -1
            elif proc.poll() is not None:
                break

        if proc.poll() is not None:
            remainder = proc.stdout.read()
            if remainder:
                print(remainder, end="")
            break

        now = time.time()
        if now - last_heartbeat >= heartbeat_sec:
            done, total = _marker_progress(run_root)
            parts = [
                f"elapsed={int(now - started)}s",
                f"marker_progress={done}/{total if total else '?'}",
            ]
            if active_run:
                parts.append(f"active={active_run}")

            if active_log_path is not None:
                if active_log_path.exists():
                    current_size = active_log_path.stat().st_size
                    if last_log_size >= 0:
                        delta = current_size - last_log_size
                        parts.append(
                            f"log_size={current_size/1024:.1f}KB (delta={delta/1024:.1f}KB)"
                        )
                    else:
                        parts.append(f"log_size={current_size/1024:.1f}KB")
                    last_log_size = current_size
                else:
                    parts.append("log=pending")

            print(f"[{label} heartbeat] " + " | ".join(parts))
            last_heartbeat = now

    rc = proc.wait()
    print(f"[{label}] process exit code: {rc}")
    print_training_activity_check(run_root, label)
    return rc

In [ ]:
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    prop = torch.cuda.get_device_properties(idx)
    print("GPU:", torch.cuda.get_device_name(idx))
    print("VRAM (GB):", round(prop.total_memory / (1024 ** 3), 2))
else:
    print("No CUDA device detected. Full medium runs will be very slow on CPU.")

In [ ]:
# Clone LOZO if needed, then enforce pinned commit for reproducibility.
if not LOZO_ROOT.exists():
    LOZO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/optsuite/LOZO", str(LOZO_ROOT)],
        check=True,
    )

subprocess.run(["git", "-C", str(LOZO_ROOT), "fetch", "--all", "--tags", "--prune"], check=True)
subprocess.run(["git", "-C", str(LOZO_ROOT), "checkout", "--detach", LOZO_PINNED_REF], check=True)

commit = subprocess.check_output(["git", "-C", str(LOZO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("LOZO commit:", commit)
print("Pinned ref OK:", commit == LOZO_PINNED_REF)
print("medium_models exists:", MEDIUM_DIR.exists())

---
## §1 — Verify LOZO Medium Pipeline

LOZO upstream entry points used in this notebook:
- `medium_models/lozo.sh`
- `medium_models/run_lozo.py`
- `medium_models/src/LOZOtrainer.py`
- `medium_models/tools/generate_k_shot_data.py`

This notebook does not duplicate training internals. It orchestrates upstream scripts and parses outputs.

In [ ]:
required = [
    MEDIUM_DIR / "lozo.sh",
    MEDIUM_DIR / "run_lozo.py",
    MEDIUM_DIR / "src" / "LOZOtrainer.py",
    MEDIUM_DIR / "src" / "modeling_roberta.py",
    MEDIUM_DIR / "tools" / "generate_k_shot_data.py",
]

for p in required:
    print(f"{p}:", "OK" if p.exists() else "MISSING")

# Fast preflight: binaries, python packages (incl loralib), required upstream files.
cmd_preflight = [
    sys.executable,
    "-m",
    "src.scripts.check_lozo_medium_env",
    "--lozo-root", str(LOZO_ROOT),
    "--tasks", "SST-2,RTE",
    "--k-values", "16",
    "--seeds", "42",
]
print(" ".join(cmd_preflight))
subprocess.run(cmd_preflight, check=True, cwd=REPO_ROOT)

---
## §2 — Data Prep (K-Shot Splits)

LOZO medium-model experiments use LM-BFF style k-shot data. Upstream defaults in the paper:
- `K in {16, 512}`
- Seeds: `{13, 21, 42, 87, 100}`
- Dataset root under `medium_models/data/k-shot-1k-test`

Run the next cell once per machine.

In [ ]:
import tarfile
import urllib.request

# Ensure LOZO medium data directory exists.
data_dir = MEDIUM_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# Download datasets if needed.
download_script = data_dir / "download_dataset.sh"
original_dir = data_dir / "original"

if not (original_dir / "SST-2" / "train.tsv").exists():
    if download_script.exists():
        subprocess.run(["bash", "download_dataset.sh"], cwd=data_dir, check=True)
    else:
        dataset_url = "https://nlp.cs.princeton.edu/projects/lm-bff/datasets.tar"
        archive_path = data_dir / "datasets.tar"
        print(f"Downloading {dataset_url} -> {archive_path}")
        urllib.request.urlretrieve(dataset_url, archive_path)

        # Upstream tar is expected to unpack under data/original/...
        with tarfile.open(archive_path) as tar:
            tar.extractall(path=data_dir)

# Normalize accidental nested layout: data/original/original/<TASK>/...
nested_original = original_dir / "original"
if nested_original.exists() and not (original_dir / "SST-2" / "train.tsv").exists():
    for child in nested_original.iterdir():
        target = original_dir / child.name
        if target.exists():
            continue
        shutil.move(str(child), str(target))

# Validate expected upstream files before generating k-shot splits.
required_train = original_dir / "SST-2" / "train.tsv"
if not required_train.exists():
    raise FileNotFoundError(
        f"Missing expected dataset file: {required_train}. "
        "Please re-run this cell after checking data download permissions."
    )

# Generate splits for K=16 and K=512 using upstream tool defaults.
for k in [16, 512]:
    cmd = [
        sys.executable,
        "tools/generate_k_shot_data.py",
        "--mode", "k-shot-1k-test",
        "--k", str(k),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=MEDIUM_DIR, check=True)

print("Data prep complete.")

In [ ]:
probe_paths = [
    MEDIUM_DIR / "data" / "k-shot-1k-test" / "SST-2" / "16-42",
    MEDIUM_DIR / "data" / "k-shot-1k-test" / "RTE" / "512-13",
]
for p in probe_paths:
    print(p, "->", "OK" if p.exists() else "MISSING")

---
## §3 — Smoke Suite (LOZO-only)

We run a short, resume-safe LOZO smoke suite using `src.scripts.run_lozo_medium_suite`.

Smoke profile defaults:
- Tasks: `SST-2`, `RTE`
- `K=16`
- Seed: `42`
- Method: `lozo`
- Short schedule: `--smoke-steps 1000 --smoke-eval-steps 100`

This validates environment, paths, logs, completion markers, and plot artifacts before full runs.

In [ ]:
SMOKE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

cmd_smoke_dry = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--expected-lozo-ref", LOZO_PINNED_REF,
    "--run-root", str(SMOKE_RUN_ROOT),
    "--profile", "smoke",
    "--methods", "lozo",
    "--dry-run",
]
print(" ".join(cmd_smoke_dry))
subprocess.run(cmd_smoke_dry, check=True, cwd=REPO_ROOT)

manifest_path = SMOKE_RUN_ROOT / "manifest.json"
print("Manifest exists:", manifest_path.exists())
if manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    print("Planned smoke runs:", m.get("num_runs"))
    print("Methods:", sorted({r["method"] for r in m.get("runs", [])}))

In [ ]:
# Toggle this to launch smoke runs.
RUN_SMOKE = False

cmd_smoke = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--expected-lozo-ref", LOZO_PINNED_REF,
    "--run-root", str(SMOKE_RUN_ROOT),
    "--profile", "smoke",
    "--methods", "lozo",
]

if RUN_SMOKE:
    rc = run_suite_with_activity(
        cmd=cmd_smoke,
        run_root=SMOKE_RUN_ROOT,
        label="smoke",
        heartbeat_sec=30,
    )
    if rc != 0:
        print("[smoke] command returned non-zero. Check hints/log_path above.")
else:
    print("RUN_SMOKE is False. Set RUN_SMOKE=True to start smoke training.")
    print("Command preview:")
    print(" ".join(cmd_smoke))

In [ ]:
smoke_summary_path = SMOKE_RUN_ROOT / "summary.json"
if smoke_summary_path.exists():
    print_training_activity_check(SMOKE_RUN_ROOT, "smoke")
    smoke = json.loads(smoke_summary_path.read_text())
    print("Smoke counts:", smoke.get("counts", {}))
    for rec in smoke.get("records", [])[:6]:
        print(
            rec.get("method"), rec.get("task"),
            f"k={rec.get('k')} seed={rec.get('seed')}",
            "status=", rec.get("status"),
            "metrics=", rec.get("metrics", {}),
        )
else:
    print("No smoke summary yet. Run the execution cell first.")

---
## §4 — Full Medium-Scale Suite

Full profile defaults map to the paper-style medium setup:
- Tasks: `SST-2`, `sst-5`, `SNLI`, `MNLI`, `RTE`, `trec`
- K values: `16`, `512`
- Seeds: `13`, `21`, `42`, `87`, `100`
- Method: `lozo`
- Upstream schedule defaults from scripts: `STEP=100000`, `EVAL_STEP=10000`

Run matrix size (default): `6 tasks x 2 K x 5 seeds x 1 method = 60 runs`.

In [ ]:
FULL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

cmd_full_dry = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--expected-lozo-ref", LOZO_PINNED_REF,
    "--run-root", str(FULL_RUN_ROOT),
    "--profile", "full",
    "--methods", "lozo",
    "--dry-run",
]
print(" ".join(cmd_full_dry))
subprocess.run(cmd_full_dry, check=True, cwd=REPO_ROOT)

full_manifest = FULL_RUN_ROOT / "manifest.json"
if full_manifest.exists():
    m = json.loads(full_manifest.read_text())
    print("Planned full runs:", m.get("num_runs"))
    print("Methods:", sorted({r['method'] for r in m['runs']}))
    print("Tasks:", sorted({r['task'] for r in m['runs']}))

In [ ]:
# Toggle this to launch full runs.
RUN_FULL = False

cmd_full = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--expected-lozo-ref", LOZO_PINNED_REF,
    "--run-root", str(FULL_RUN_ROOT),
    "--profile", "full",
    "--methods", "lozo",
]

if RUN_FULL:
    rc = run_suite_with_activity(
        cmd=cmd_full,
        run_root=FULL_RUN_ROOT,
        label="full",
        heartbeat_sec=60,
    )
    if rc != 0:
        print("[full] command returned non-zero. Check hints/log_path above.")
else:
    print("RUN_FULL is False. Set RUN_FULL=True to start full training.")
    print("Command preview:")
    print(" ".join(cmd_full))

---
## §5 — Aggregate LOZO Results and Plot Memory/Performance

Aggregation reads `summary.json` produced by the orchestration script and computes per-task LOZO summaries.

By default, this section reads from `FULL_RUN_ROOT/summary.json`. You can point it to smoke outputs for quick testing.

In [ ]:
SUMMARY_SOURCE = FULL_RUN_ROOT / "summary.json"
# For quick debugging, you can switch to: SUMMARY_SOURCE = SMOKE_RUN_ROOT / "summary.json"

if SUMMARY_SOURCE.exists():
    print_training_activity_check(SUMMARY_SOURCE.parent, SUMMARY_SOURCE.parent.name)


def choose_metric(metrics: dict[str, float]) -> tuple[str | None, float | None]:
    if not metrics:
        return None, None
    ordered = sorted(metrics.items())
    for key, value in ordered:
        if "acc" in key.lower() or "f1" in key.lower() or "mcc" in key.lower():
            return key, value
    return ordered[0]


def load_lozo_rows(summary_path: Path) -> tuple[list[dict], list[dict], dict]:
    if not summary_path.exists():
        return [], [], {}

    payload = json.loads(summary_path.read_text())
    rows = []
    mem_rows = []
    for rec in payload.get("records", []):
        if rec.get("method") != "lozo":
            continue
        if rec.get("status") not in {"completed", "skipped"}:
            continue

        metric_name, metric_val = choose_metric(rec.get("metrics", {}))
        if metric_val is not None:
            rows.append(
                {
                    "task": rec.get("task"),
                    "k": int(rec.get("k")),
                    "seed": int(rec.get("seed")),
                    "metric_name": metric_name,
                    "metric": float(metric_val),
                    "status": rec.get("status"),
                    "result_dir": rec.get("result_dir"),
                    "log_path": rec.get("log_path"),
                }
            )

        peak = rec.get("gpu_memory_mb_peak")
        if peak is not None:
            mem_rows.append(
                {
                    "task": rec.get("task"),
                    "k": int(rec.get("k")),
                    "seed": int(rec.get("seed")),
                    "peak_mb": float(peak),
                }
            )

    return rows, mem_rows, payload


def aggregate_lozo(rows: list[dict]) -> list[dict]:
    buckets: dict[tuple[str, int], list[float]] = defaultdict(list)
    for r in rows:
        buckets[(r["task"], r["k"])].append(r["metric"])

    out = []
    for (task, k), vals in sorted(buckets.items()):
        mean = sum(vals) / len(vals)
        var = sum((v - mean) ** 2 for v in vals) / len(vals)
        out.append(
            {
                "task": task,
                "k": k,
                "n": len(vals),
                "mean": mean,
                "std": var ** 0.5,
            }
        )
    return out


rows, mem_rows, raw_summary = load_lozo_rows(SUMMARY_SOURCE)
agg = aggregate_lozo(rows)
print(f"summary path: {SUMMARY_SOURCE}")
print("LOZO SHA:", raw_summary.get("lozo_sha"))
print(f"lozo records with metrics: {len(rows)}")
print(f"lozo aggregated groups: {len(agg)}")
print(f"lozo records with gpu memory: {len(mem_rows)}")

In [ ]:
if not agg:
    print("No LOZO aggregated results yet. Run smoke/full suite first.")
else:
    print(f"{'Task':<10} {'K':>4} {'n':>3} {'mean':>10} {'std':>10}")
    print("-" * 45)
    for r in agg:
        print(f"{r['task']:<10} {r['k']:>4} {r['n']:>3} {r['mean']:>10.4f} {r['std']:>10.4f}")

if mem_rows:
    print("\nPeak GPU memory per LOZO run")
    print(f"{'Task':<10} {'K':>4} {'Seed':>6} {'PeakMB':>10}")
    print("-" * 36)
    for r in sorted(mem_rows, key=lambda x: (x['task'], x['k'], x['seed'])):
        print(f"{r['task']:<10} {r['k']:>4} {r['seed']:>6} {r['peak_mb']:>10.1f}")

In [ ]:
if not agg:
    print("No LOZO data to plot yet.")
else:
    labels = [f"{r['task']} (k={r['k']})" for r in agg]
    means = [r["mean"] for r in agg]
    stds = [r["std"] for r in agg]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: final performance by task/K
    x = list(range(len(labels)))
    axes[0].bar(x, means, yerr=stds, capsize=4)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=35, ha="right")
    axes[0].set_ylabel("Final metric")
    axes[0].set_title("LOZO final metric by task")

    # Plot 2: peak GPU memory (if available)
    if mem_rows:
        mem_labels = [f"{r['task']}-k{r['k']}-s{r['seed']}" for r in mem_rows]
        mem_vals = [r["peak_mb"] for r in mem_rows]
        xm = list(range(len(mem_vals)))
        axes[1].plot(xm, mem_vals, marker="o")
        axes[1].set_xticks(xm)
        axes[1].set_xticklabels(mem_labels, rotation=45, ha="right")
        axes[1].set_ylabel("Peak GPU memory (MB)")
        axes[1].set_title("LOZO peak GPU memory by run")
    else:
        axes[1].text(0.5, 0.5, "No GPU memory samples in summary", ha="center", va="center")
        axes[1].set_axis_off()

    fig.tight_layout()
    out_fig = FULL_RUN_ROOT / "lozo_only_metric_memory.png"
    out_fig.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_fig, dpi=180)
    plt.show()
    print("Saved:", out_fig)

# Optional: regenerate same figures via standalone script for reproducibility.
cmd_plot = [
    sys.executable,
    "-m",
    "src.scripts.plot_lozo_medium_suite",
    "--summary", str(SUMMARY_SOURCE),
]
print("Scripted plotting command:")
print(" ".join(cmd_plot))

---
## §6 — Reproducibility, Runtime, and Recovery

### Resume behavior
- The runner checks completion markers (`test_results_<TASK>.txt`) and skips finished runs.
- Re-run the same command with the same `--run-root` to resume.
- Use `--force` to re-run completed configurations.

### Runtime guidance (single 80GB GPU)
- **Smoke profile**: typically minutes to a couple of hours depending on download/cache state.
- **Full profile (60 LOZO runs)**: can take multiple days. Run in stages and monitor `summary.json`.
- Keep a persistent cache for model/data weights between machine restarts.

### Practical recovery commands
```bash
# Resume smoke (LOZO only)
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --expected-lozo-ref 5d1ade5bf41846471d33ba526f877829b9a19856 \
  --run-root results/lozo_medium_prime80gb/smoke \
  --profile smoke \
  --methods lozo

# Resume full suite (LOZO only)
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --expected-lozo-ref 5d1ade5bf41846471d33ba526f877829b9a19856 \
  --run-root results/lozo_medium_prime80gb/full \
  --profile full \
  --methods lozo

# Re-run everything in full profile
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --expected-lozo-ref 5d1ade5bf41846471d33ba526f877829b9a19856 \
  --run-root results/lozo_medium_prime80gb/full \
  --profile full \
  --methods lozo \
  --force
```

### Common issues
- `data/k-shot-1k-test/...` missing: run the data-prep section again.
- Missing `loralib`: run `uv sync` and rerun preflight.
- Missing CUDA or OOM: reduce batch size (`--lozo-bs`) or run a smaller subset (`--tasks`, `--seeds`, `--k-values`).
- Interrupted jobs: rerun same command; completed runs are skipped automatically.